# Tome 12, Chapter 7: Quadratic forms and the local–global principle (Python companion)

Exercises first, solutions at the end. We implement the Hilbert symbol from Theorem 7.3 and check it against PARI, test the product formula, turn Theorem 7.10 into a decision procedure for "does this form have a rational zero?", watch the exceptions $4^a(8b+7)$ appear among sums of three squares, and end with a cryptographic use of Gauss's three-squares theorem: proving that a hidden number is non-negative. Requires `cypari2`.

In [ ]:
import math, random, time
import cypari2
pari = cypari2.Pari()
pari.allocatemem(10**9, silent=True)
random.seed(2026)

## Exercises

**P7.1 (★) The Hilbert symbol by hand.** Implement $(a,b)_v$ for non-zero integers $a,b$ and $v=\infty$ or a prime, using the formulas of Theorem 7.3. Compare with PARI's `hilbert(a, b, p)` (where `p=0` means $\infty$) on many random pairs, and check the product formula $\prod_v(a,b)_v=1$ (Theorem 7.5). Which places can contribute a $-1$?

**P7.2 (★★) A local–global decision procedure.** Using the invariants $d$ and $\varepsilon$ and Theorem 7.8(a), write `isotropic_at(a, v)` for a diagonal form $\langle a_1,\dots,a_n\rangle$ with integer coefficients, and `isotropic_Q(a)` using Hasse–Minkowski. Which finitely many places do you actually need to check? Compare with PARI's `qfsolve`, which returns either a rational zero or a place where there is none.

**P7.3 (★) The hook.** The conic $x^2-6y^2-21z^2=0$ of the chapter's opening has real points. At which places $v$ does it have no point over $\Q_v$? Compute $(6,21)_v$ at every relevant place, explain why the number of bad places is even, and see what `qfsolve` answers.

**P7.4 (★★) Three squares.** For $n\le 10^5$, compute which $n$ are sums of three squares (a sieve over $x\le y\le z$ is fine), check that the exceptions are exactly the $4^a(8b+7)$, and compare the proportion of exceptions with the limit $1/6$. Then compute $r_3(n)$ (the number of representations with signs and order) for $n\le 2000$ and plot $r_3(n)/\sqrt n$: does it look bounded? bounded below on non-exceptions?

**P7.5 (★★★) CS application: range proofs from three squares.** In privacy-preserving protocols (e-cash, confidential transactions, anonymous credentials) a prover holds a secret integer $y$ hidden inside a commitment, and must convince a verifier that $y\ge0$ without revealing it. Commitments à la Pedersen are additively homomorphic, so it is easy to prove polynomial identities about committed numbers, but not inequalities. Groth's trick: *$y\ge0$ if and only if $4y+1$ is a sum of three squares*.
1. Prove the equivalence from Theorem 7.18.
2. The prover must actually *find* $x_1,x_2,x_3$ with $x_1^2+x_2^2+x_3^2=4y+1$. Implement a randomised algorithm: pick $x_1$ even at random until $m=4y+1-x_1^2$ is a prime (then $m\equiv1\pmod4$), and write $m$ as a sum of two squares by the Hermite–Serret method (find $t$ with $t^2\equiv-1\pmod m$, then run Euclid's algorithm on $(m,t)$ and stop at the first remainder below $\sqrt m$).
3. Time it for $y$ of $64$, $256$ and $1024$ bits. Why is the expected number of trials about $\log y$, at least heuristically?

## Solutions

### P7.1
**How to find it.** Write $a=p^\alpha u$, $b=p^\beta v$ with $u,v$ units. For odd $p$, $(a,b)_p=(-1)^{\alpha\beta\epsilon(p)}\bigl(\frac up\bigr)^\beta\bigl(\frac vp\bigr)^\alpha$; for $p=2$, $(a,b)_2=(-1)^{\epsilon(u)\epsilon(v)+\alpha\omega(v)+\beta\omega(u)}$ with $\epsilon(u)=\frac{u-1}2$, $\omega(u)=\frac{u^2-1}8$ modulo $2$. At $\infty$ the symbol is $-1$ exactly when both are negative.

In [ ]:
def split(a, p):
    k = 0
    while a % p == 0:
        a //= p; k += 1
    return k, a

def legendre(u, p):
    return 1 if pow(u % p, (p - 1) // 2, p) == 1 else -1

def hilbert(a, b, p):
    if p == 0:
        return -1 if (a < 0 and b < 0) else 1
    al, u = split(a, p); be, v = split(b, p)
    if p == 2:
        eps = lambda x: ((x - 1) // 2) % 2
        om = lambda x: ((x * x - 1) // 8) % 2
        e = eps(u) * eps(v) + al * om(v) + be * om(u)
    else:
        e = al * be * ((p - 1) // 2)
        s = (legendre(u, p) if be % 2 else 1) * (legendre(v, p) if al % 2 else 1)
        return (-1) ** (e % 2) * s
    return (-1) ** (e % 2)

def places(*nums):
    ps = {2}
    for x in nums:
        ps |= {int(q) for q in pari.factor(abs(x))[0]}
    return [0] + sorted(ps)

bad = 0
for _ in range(3000):
    a = random.choice([-1, 1]) * random.randint(1, 10**6)
    b = random.choice([-1, 1]) * random.randint(1, 10**6)
    prod = 1
    for p in places(a, b):
        h = hilbert(a, b, p)
        bad += h != int(pari.hilbert(a, b, p))
        prod *= h
    bad += prod != 1
print("disagreements with PARI or product formula:", bad)

No disagreement. Only $\infty$, $2$ and the primes dividing $ab$ can carry a $-1$: at an odd prime where both are units the formula gives $1$. So the product in Theorem 7.5 is a finite product, and the code only loops over those places.

### P7.2
**How to find it.** Over $\Q_p$ (Theorem 7.8(a)): $n=2$ isotropic iff $-d$ is a square; $n=3$ iff $(-1,-d)_p=\varepsilon$; $n=4$ iff $d$ is not a square or $\varepsilon=(-1,-1)_p$; $n\ge5$ always. Over $\R$: iff indefinite. For $n\ge3$, at an odd prime where all $a_i$ are units, $d$ is a unit and $\varepsilon=1=(-1,-d)_p$, so the form is isotropic: only $\infty$, $2$ and the primes dividing some $a_i$ need checking. For $n=2$ we just ask whether $-a_1a_2$ is a square in $\Q$.

In [ ]:
def is_square_Qp(x, p):
    k, u = split(x, p)
    if k % 2:
        return False
    if p == 2:
        return u % 8 == 1
    return legendre(u, p) == 1

def eps_inv(a, p):
    e = 1
    for i in range(len(a)):
        for j in range(i + 1, len(a)):
            e *= hilbert(a[i], a[j], p)
    return e

def isotropic_at(a, p):
    n, d = len(a), math.prod(a)
    if p == 0:
        return min(a) < 0 < max(a)
    if n == 2:
        return is_square_Qp(-d, p)
    e = eps_inv(a, p)
    if n == 3:
        return hilbert(-1, -d, p) == e
    if n == 4:
        return (not is_square_Qp(d, p)) or e == hilbert(-1, -1, p)
    return True

def isotropic_Q(a):
    if len(a) == 2:
        return math.isqrt(-a[0] * a[1]) ** 2 == -a[0] * a[1] if -a[0] * a[1] >= 0 else False
    return all(isotropic_at(a, p) for p in places(*a))

def pari_says(a):
    r = pari.qfsolve(pari.matdiagonal(a))
    return str(pari.type(r)) == "t_COL"

mism, count = 0, {True: 0, False: 0}
for _ in range(1500):
    n = random.choice([2, 3, 3, 4, 4, 5])
    a = [random.choice([-1, 1]) * random.randint(1, 60) for _ in range(n)]
    ours = isotropic_Q(a)
    count[ours] += 1
    mism += ours != pari_says(a)
print("mismatches with qfsolve:", mism, "| isotropic / anisotropic:", count[True], count[False])

Our hundred-line implementation of Hasse–Minkowski agrees with PARI on every form. The procedure is the theorem made executable: finitely many places, each decided by a sign or a Legendre symbol. Notice what it does *not* do: search for a solution. Deciding existence is cheap; finding the solution is the job of `qfsolve` (Legendre's descent, P7.3).

### P7.3
**How to find it.** The form is $\langle1,-6,-21\rangle$; it is isotropic iff $6$ is a norm from $\Q(\sqrt{21})$, i.e. iff $(6,21)_v=1$ for all $v$. Compute.

In [ ]:
for p in places(6, 21):
    print("v =", "oo" if p == 0 else p, " (6,21)_v =", hilbert(6, 21, p))
print("qfsolve:", pari.qfsolve(pari.matdiagonal([1, -6, -21])))

The symbol is $-1$ at $2$ and at $7$ (two failures, as the product formula demands: an odd number of failures is impossible), so the conic has points over $\R$ and every $\Q_p$, $p\ne2,7$, but none over $\Q$. PARI reports one of the obstructing places. The opening of the chapter in one line: *local failures come in pairs*.

### P7.4
**How to find it.** Mark every $x^2+y^2+z^2\le N$. The density of $\{4^a(8b+7)\}$ is $\frac18(1+\frac14+\frac1{16}+\cdots)=\frac18\cdot\frac43=\frac16$.

In [ ]:
N = 10**5
rep = bytearray(N + 1)
r = math.isqrt(N)
for x in range(r + 1):
    for y in range(x, r + 1):
        s = x * x + y * y
        if s > N: break
        for z in range(y, r + 1):
            t = s + z * z
            if t > N: break
            rep[t] = 1
def exceptional(n):
    while n % 4 == 0: n //= 4
    return n % 8 == 7
ok = all((rep[n] == 1) != exceptional(n) for n in range(1, N + 1))
print("exceptions are exactly 4^a(8b+7):", ok)
print("proportion of exceptions:", sum(1 for n in range(1, N + 1) if not rep[n]) / N, " vs 1/6 =", 1 / 6)

In [ ]:
import matplotlib.pyplot as plt
M = 2000
r3 = [0] * (M + 1)
R = math.isqrt(M)
for x in range(-R, R + 1):
    for y in range(-R, R + 1):
        s = x * x + y * y
        if s > M: continue
        for z in range(-R, R + 1):
            t = s + z * z
            if t <= M: r3[t] += 1
ns = [n for n in range(1, M + 1) if not exceptional(n)]
plt.figure(figsize=(8, 3.5))
plt.scatter(ns, [r3[n] / math.sqrt(n) for n in ns], s=2)
plt.xlabel("n"); plt.ylabel("r3(n)/sqrt(n)"); plt.title("Representations by three squares")
plt.show()
print("n = 4^k: r3 =", [r3[4**k] for k in range(6)])

The cloud is not bounded below by a positive constant: $r_3(4^k)=6$ for every $k$ (the only representations of $4^k$ are $(\pm2^k,0,0)$ and permutations, by descent modulo $4$), and the points with $4\mid n$ form lower layers. On squarefree $n\not\equiv7\pmod 8$, Gauss showed that $r_3(n)$ is a constant multiple of a class number $h(-4n)$ (up to small factors), and Siegel's theorem $h(-D)=D^{1/2+o(1)}$ explains the $\sqrt n$ scale. This is the first glimpse of the bridge between quadratic forms and class numbers that Chapter 14 (modular forms) makes explicit.

### P7.5
**How to find it.** (1) If $y\ge0$, then $4y+1\equiv1\pmod 4$, so it is neither divisible by $4$ nor $\equiv7\pmod8$, and Theorem 7.18 applies. Conversely, if $4y+1=x_1^2+x_2^2+x_3^2\ge0$ then $y\ge-\frac14$, and $y$ is an integer, so $y\ge0$. (2) If $m\equiv1\pmod 4$ is prime, a square root $t$ of $-1$ is $g^{(m-1)/4}$ for any non-residue $g$; the first remainder below $\sqrt m$ in the Euclidean algorithm on $(m,t)$, together with the next one, gives $m=c^2+d^2$ (Hermite–Serret; Cornacchia's algorithm in general). (3) By the prime number theorem in progressions, numbers of size $m$ that are $\equiv1\pmod4$ are prime with probability about $2/\log m$, and the values $4y+1-x_1^2$ behave, heuristically, like random such numbers; this is not a theorem, which is why practical systems accept a randomised algorithm with a fast expected running time.

In [ ]:
def sqrt_minus_one(p):
    while True:
        g = random.randrange(2, p - 1)
        t = pow(g, (p - 1) // 4, p)
        if t * t % p == p - 1:
            return t

def two_squares_prime(p):
    if p == 2:
        return 1, 1
    a, b = p, sqrt_minus_one(p)
    r = math.isqrt(p)
    while b > r:
        a, b = b, a % b
    c = b
    d = math.isqrt(p - c * c)
    assert c * c + d * d == p
    return c, d

def three_squares_4y1(y):
    m0 = 4 * y + 1
    R = math.isqrt(m0)
    trials = 0
    while True:
        trials += 1
        x1 = 2 * random.randint(0, R // 2)
        m = m0 - x1 * x1
        if m == 1:
            return (x1, 1, 0), trials
        if m > 1 and pari.ispseudoprime(m):
            c, d = two_squares_prime(m)
            return (x1, c, d), trials

for bits in [64, 256, 1024]:
    ts, tr = [], []
    for _ in range(60 if bits < 1024 else 20):
        y = random.getrandbits(bits)
        t0 = time.perf_counter()
        (a, b, c), k = three_squares_4y1(y)
        ts.append(time.perf_counter() - t0); tr.append(k)
        assert a * a + b * b + c * c == 4 * y + 1
    print(f"{bits:5d} bits: mean trials {sum(tr)/len(tr):7.1f} (log y ≈ {bits*math.log(2):6.1f}),"
          f" mean time {1000*sum(ts)/len(ts):8.2f} ms")

The number of trials grows linearly with the bit length, like a constant times $\log y$, here between about $\frac12$ and $1$ (the condition $m\equiv1\pmod4$ doubles the chance of primality, while the residues of $4y+1$ modulo $3,5,7,\dots$ can forbid some $x_1$ and push the constant up; and the number of trials is geometrically distributed, so averages over a few dozen runs fluctuate by $20\%$), and each trial costs one probable-prime test. A $1024$-bit witness takes a fraction of a second; a certified primality proof (`isprime`) would cost far more, and is unnecessary, since the verifier only checks the final identity. In the protocol the prover then commits to $x_1,x_2,x_3$ and proves the *polynomial identity* $x_1^2+x_2^2+x_3^2=4y+1$, which homomorphic commitments handle well. Gauss's theorem turns an inequality, which commitments cannot see, into an equation, which they can.

**Reference.** J. Groth, *Non-interactive zero-knowledge arguments for voting*, ACNS 2005 (the three-squares trick); M. O. Rabin and J. Shallit, *Randomized algorithms in number theory*, Comm. Pure Appl. Math. 39 (1986) (sums of squares in random polynomial time).